# NB3 · Modeli kurmak ve ölçmek

**Üretken Yapay Zekâ Araçları ile Klinik Karar Destek Sistemleri Geliştirilmesi**  
Sağlık Bilimlerinde Teknoloji ve Yapay Zekâ Okuryazarlığı Eğitimi · Akdeniz Üniversitesi · 18 Eylül 2026

Prof. Dr. Utku Köse · Süleyman Demirel Üniversitesi, Bilgisayar Mühendisliği Bölümü  
Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM) Müdürü · utkukose@sdu.edu.tr

---


## Bu defterde ne yapılıyor

Model kurmak bu defterin en kısa işidir; birkaç satır sürer. Defterin geri kalanı o
modelin ne kadar işe yaradığını ölçmeye ayrılmıştır.

Ölçmenin uzun sürmesinin sebebi şudur: Bir modelin iyi görünmesi kolaydır, gerçekten iyi
olup olmadığını anlamak zordur. Bu defterde her ikisini de göreceksiniz.


## Kurulum


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
urllib.request.urlretrieve(f'{REPO}/workshop/cdss_kit.py', 'cdss_kit.py')

import cdss_kit as kit
kit.LANG = 'tr'

print('Hazır.')


---

## Önceki defterden gelen kod

Bir önceki defterin sonunda toplanan bloğun tamamını aşağıdaki hücreye yapıştırınız.
İlk satırdaki `#@cdss` işaretini silmeyiniz; o blok bu defterin sonunda yeniden
toplanacak ve bir sonrakine taşınacaktır.

Blok çalıştığında önceki defterlerde yazdığınız her şey yeniden kurulur. İnternetten
veri okuyan satırlar varsa bu hücre birkaç saniye sürebilir.


In [ ]:
#@cdss onceki_defter
# Ürettiğiniz kodu bu satırın altına yapıştırınız.


### Kontrol · Gelen kod


In [ ]:
kit.check_defined('X_egitim', 'X_sinama', 'y_egitim', 'y_sinama',
                  'egitim', 'sinama', 'RASTGELE_TOHUM')


---

## Adım 1 · Modeli kurmak ve öğretmek

Model, veriden örüntü çıkaran bir hesap yöntemidir. Basit bir modelle başlıyoruz.

Basitle başlamanın sebebi başarım değil, üç özelliktir. Basit modelin neye baktığı
okunabilir; olasılık üretir; az veriyle ezberleme eğilimi düşüktür. Daha karmaşık bir
model bunu aşarsa aşsın, ölçüt elimizde durur.

Bir konuya dikkat ediniz. Hedef durum kohortta azınlıktaysa model kolay yolu seçip
herkese olumsuz diyebilir. İstemde bunu engellemek için sınıfların ağırlıklandırılmasını
isteyeceğiz.


### İstem 1

```
X_egitim ve y_egitim ile bir model kur ve öğret.

Basit ve yorumlanabilir bir sınıflandırma yöntemi kullan. Hedef durum azınlıktaysa
modelin çoğunluğa kaymasını engelle; sınıfları dengeli ağırlıklandır.

Tekrar üretilebilirlik için RASTGELE_TOHUM değerini kullan. Şimdilik ayar araması
yapma; bu ilk model aşılmak için kuruluyor.

Modeli model adıyla sakla. Ekrana hangi yöntemi seçtiğini ve neden bu problem için uygun
bir başlangıç olduğunu iki cümleyle yaz.

BİÇİM
Tek bir Python hücresi yaz. Satırların yanına Türkçe açıklama ekle. Kısa ve
okunabilir yaz; kodu Python bilmeyen biri takip edebilmeli. Hazır kısayollar yerine
adımları açıkça göster. Kodun altına, ne yaptığını üç cümleyle sade bir dille özetle.

BEKLENEN SONUÇ
model adında öğretilmiş bir model hazır olmalı.
Model, bir hastanın hedef duruma girme olasılığını verebilmeli; yalnızca 0 veya 1
etiketi değil, 0 ile 1 arasında bir sayı üretebilmeli.
```


In [ ]:
#@cdss model
# Ürettiğiniz kodu bu satırın altına yapıştırınız.


### Kontrol 1


In [ ]:
kit.check_model('model', sample=X_sinama)


### Python notu · Nesne ve yöntem

Gelen kodda `model = LogisticRegression(...)` gibi bir satır ve ardından `model.fit(...)`
göreceksiniz. Burada iki kavram var.

İlk satır bir **nesne** üretir. Nesne, hem bilgi hem de o bilgiyle yapılabilecek işleri
bir arada tutan bir yapıdır. Bir hasta dosyası gibi düşünebilirsiniz: İçinde veriler
vardır ve o dosyayla yapılabilecek belirli işlemler tanımlıdır.

`model.fit(...)` ise nesnenin bir **yöntemini** çağırır. Noktadan sonraki ad, o nesnenin
yapabildiği bir iştir. `fit` öğrenmeyi, `predict` tahmin etmeyi, `predict_proba` olasılık
üretmeyi sağlar.

Bu yapı makine öğrenmesi kütüphanelerinin tamamında aynıdır. Yöntemi değiştirseniz de
`fit` ve `predict` adları değişmez; bu yüzden bir modeli başkasıyla değiştirmek kolaydır.


---

## Adım 2 · Tahmin üretmek

Model artık sınama grubundaki hastalar için tahmin üretebilir.

Tahminin sınıf etiketi değil olasılık olması gerekir. Sınıf etiketi üreten bir model,
eşiği kendi varsayılanına sabitlemiştir; genellikle yüzde 50. O varsayılan klinik bir
karar değildir.

Eşiği siz seçersiniz ve seçim klinik gerekçeye dayanır. Kaçırmanın pahalı olduğu bir
problemde eşik düşürülür, daha çok uyarı üretilir. Yanlış alarmın pahalı olduğu bir
problemde tersi yapılır. Bunu üçüncü adımda göreceksiniz.


### İstem 2

```
Öğretilmiş model ile sınama grubundaki hastalar için tahmin üret.

Sınıf etiketi değil olasılık iste: Her hasta için hedef duruma girme olasılığı, 0 ile 1
arasında bir sayı olsun.

Sonucu olasilik adıyla sakla. Ekrana en düşük, en yüksek ve ortalama olasılığı yaz.

BİÇİM
Tek bir Python hücresi yaz. Satırların yanına Türkçe açıklama ekle. Kısa ve
okunabilir yaz; kodu Python bilmeyen biri takip edebilmeli. Hazır kısayollar yerine
adımları açıkça göster. Kodun altına, ne yaptığını üç cümleyle sade bir dille özetle.

BEKLENEN SONUÇ
olasilik adında bir sayı dizisi hazır olmalı.
Uzunluğu sınama grubundaki hasta sayısına eşit olmalı.
Değerler 0 ile 1 arasında olmalı ve hepsi aynı olmamalı.
```


In [ ]:
#@cdss tahmin
# Ürettiğiniz kodu bu satırın altına yapıştırınız.


### Kontrol 2


In [ ]:
kit.check_numbers(olasilik, name='olasilik', count=len(y_sinama), low=0.0, high=1.0)


---

## Adım 3 · Dürüst ölçüm

Şimdi asıl iş. Bir modelin başarımı tek bir sayıyla anlatılamaz; altı başlık birlikte
bakılır.

**Doğruluk yanıltır.** Hedef durum hastaların yüzde onunda görülüyorsa, herkese olumsuz
diyen bir kural yüzde 90 doğruluk verir. Doğruluk bu durumda modelin ne yaptığını değil,
durumun ne kadar seyrek olduğunu ölçer.

**Ayrım gücü tek başına yetmez.** Modelin hastaları doğru sıralaması ile verdiği
olasılığın gerçeğe karşılık gelmesi ayrı şeylerdir. Eşik belirleyecekseniz ikincisi daha
önemlidir.

**Asıl soru klinik karşılıktır.** Yüz hastada kaç uyarı çıkıyor ve kaçı doğru? Bu satır,
16 Eylül dersinde anlatılan Epic Sepsis Model örneğinin sizin modelinizdeki karşılığıdır.


### İstem 3

```
Modelin başarımını ölçen bir işlem parçası yaz. Adı basarim_olc olsun; kendisine gerçek
sonuçları, tahmin olasılıklarını ve bir eşik değerini alsın.

Şunları hesaplasın ve her birinin altına ne anlama geldiğini sade bir dille yazsın:
1. Modelin hastaları ne kadar iyi sıraladığı. Bu değeri bir güven aralığıyla birlikte
   ver; tek bir sayı yeterli değil.
2. Modelin verdiği olasılıkların gerçeğe ne kadar karşılık geldiği.
3. Verilen eşikte: Hastaları doğru yakalama oranı, boşuna uyarı vermeme oranı, uyarı
   verdiğinde haklı çıkma oranı ve uyarı vermediğinde haklı çıkma oranı.
4. Klinik karşılık: Bu eşikte her yüz hastada kaç uyarı çıkar, kaçı doğrudur, kaç vaka
   tamamen kaçırılır.
5. Cinsiyet gruplarına göre aynı ölçümler. Bir grupta hasta sayısı çok azsa sayı üretme;
   yetersiz örneklem yaz.
6. Hiçbir şey öğrenmeyen bir kuralın, yani herkese aynı cevabı veren bir kuralın aynı
   ölçütlerdeki değerleri.

Sonuçları hem ekrana yaz hem de bir sözlük olarak geri ver.

Sonra bu işlem parçasını sınama grubu üzerinde 0.50 eşiğiyle çalıştır ve sonucu sonuc
adıyla sakla.

BİÇİM
Tek bir Python hücresi yaz. Satırların yanına Türkçe açıklama ekle. Kısa ve
okunabilir yaz; kodu Python bilmeyen biri takip edebilmeli. Hazır kısayollar yerine
adımları açıkça göster. Kodun altına, ne yaptığını üç cümleyle sade bir dille özetle.

BEKLENEN SONUÇ
basarim_olc adında çalıştırılabilir bir işlem parçası olmalı.
sonuc adında bir sözlük hazır olmalı ve şu anahtarları içermeli:
  ayrim_gucu, kalibrasyon, duyarlilik, ozgulluk, pkd, nkd,
  yuz_hastada_uyari, yuz_hastada_dogru_uyari, bos_kural_dogrulugu
```


In [ ]:
#@cdss olcum
# Ürettiğiniz kodu bu satırın altına yapıştırınız.


### Kontrol 3


In [ ]:
kit.check_function('basarim_olc')

gerekli = ['ayrim_gucu', 'kalibrasyon', 'duyarlilik', 'ozgulluk', 'pkd', 'nkd',
           'yuz_hastada_uyari', 'yuz_hastada_dogru_uyari', 'bos_kural_dogrulugu']
eksik = [a for a in gerekli if a not in sonuc]
print('Sonuç sözlüğünde eksik anahtar:', eksik if eksik else 'yok')


### Doğruluk tuzağı

Aşağıdaki iki hücre hazır gelir. Birincisini çalıştırıp rakamı okuyunuz, sonra ikinciyi
çalıştırınız.


In [ ]:
import numpy as _np
dogruluk = float((( _np.asarray(olasilik) >= 0.5).astype(int) == _np.asarray(y_sinama)).mean())
print(f'Modelin doğruluğu: {dogruluk:.1%}')


In [ ]:
oran = float(_np.asarray(y_sinama).mean())
bos = max(oran, 1 - oran)
print(f'Herkese aynı cevabı veren kuralın doğruluğu: {bos:.1%}')
print()
print('Aradaki fark, modelin gerçekten kattığı değerdir.')


### Python notu · Sözlük

İstemde sonuçların bir **sözlük** olarak geri verilmesini istedik. Sözlük, her değeri bir
adla saklayan bir yapıdır: `sonuc['duyarlilik']` yazarak o değere ulaşırsınız.

Listeden farkı sıraya değil ada göre erişilmesidir. Dokuz değeri liste olarak geri
verseydik, üçüncü sıradakinin ne olduğunu hatırlamak zorunda kalırdınız. Sözlükte ad
yazar.

Klinik kodda bu tercih önemlidir: NB5'te uyumluluk raporunu yazarken bu sözlükten değer
çekeceksiniz, o zaman adların açık olması işinizi görecek.


### Sonucun okunması

Dört noktaya bakınız.

**Güven aralığı.** Aralık 0,5 değerini içeriyorsa model şanstan ayırt edilemiyor demektir;
ortadaki sayı ne olursa olsun.

**Boş kural karşılaştırması.** Modelin doğruluğu ile herkese aynı cevabı veren kuralın
doğruluğu birbirine yakınsa model kayda değer bir şey öğrenmemiştir.

**Klinik karşılık.** Yüz hastada kaç uyarı çıktığına ve kaçının doğru olduğuna bakınız.
Uyarıların çoğu yanlışsa sistem klinikte bir süre sonra kapatılır.

**Alt gruplar.** Bir grupta yetersiz örneklem yazıyorsa bu bir eksiklik değil, bir
bulgudur: Hiç sınanmamış bir grup için sistemin adil olduğu gösterilemez.

Yüz hastalık bir kümede sonucun olumsuz çıkması beklenir. Kendi kurduğunuz sistem
hakkında bu kanaate varmak, bir başkasının başarısızlığını dinlemekten farklıdır.


---

## Defter sonu · Kodun toplanması

Aşağıdaki hücre önceki defterlerden taşıdığınız kodla bu defterde eklediklerinizi tek
bir blok hâlinde toplar. Çıkan bloğun tamamını kopyalayınız; NB4 defterinin ilk
hücresine yapıştıracaksınız.

Blok ayrıca `cdss_nb3.py` adıyla kaydedilir. Colab oturumu kapandığında bu dosya silinir, bu
nedenle bloğu kendi bilgisayarınızda bir metin dosyasına da kopyalayınız.


In [ ]:
kod = kit.export(save_as='cdss_nb3.py')


## Bu defterde ne yapıldı

Sisteme model, tahmin ve ölçüm katmanları eklendi. Ölçüm tek bir sayı değil, dokuz
değerli bir sözlük üretiyor ve klinik karşılığı da içeriyor.

NB4'te modelin verdiği kararın gerekçesini üreten katman eklenecektir.
---

**Uyarı.** Bu defterde üretilen hiçbir çıktı doğrulanmış bir klinik araç değildir.
MIMIC-IV demo verisi tek bir Amerikan hastanesinden gelir ve Türkiye'deki bir yoğun
bakım popülasyonunu temsil etmez. Materyal öğretim amaçlıdır.
